# Solution D1 · Reading the decision boundary: roughness, regions and k

**Type**: Interpret

## Objective

Turn the KNN decision boundary from an image into **numbers**: for nine values of `k` compute on a grid the roughness of the boundary (fraction of adjacent cells with different classes) and the number of connected regions, and read from the data
where there is overfitting, where the model is too rigid and how large `k` must be to "simplify" the boundary.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from scipy.ndimage import label
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

X, y = make_moons(600, noise=0.3, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)
g1, g2 = np.meshgrid(np.linspace(-2.5, 3.5, 100), np.linspace(-1.8, 2.3, 100))
points = np.c_[g1.ravel(), g2.ravel()]
ks = [1, 3, 5, 9, 15, 31, 61, 121, 201]


def grid_map(k):
    """KNN prediction (k neighbors) on a 100 x 100 grid: matrix of 0s and 1s."""
    return KNeighborsClassifier(k).fit(X_tr, y_tr).predict(points).reshape(100, 100)

## Commented solution

In [ ]:
rows = {}
for k in ks:
    m = KNeighborsClassifier(k).fit(X_tr, y_tr)
    G = m.predict(points).reshape(100, 100)
    differences = (G[:, 1:] != G[:, :-1]).sum() + (G[1:, :] != G[:-1, :]).sum()                 # adjacent pairs with different classes
    rows[k] = {"train_acc": m.score(X_tr, y_tr), "test_acc": m.score(X_te, y_te),
               "roughness": differences / (G[:, 1:].size + G[1:, :].size),
               "n_regions": label(G == 0)[1] + label(G == 1)[1]}
table = pd.DataFrame(rows).T
table["n_regions"] = table["n_regions"].astype(int)
table.round(3)

In [ ]:
overfit_k = int((table["train_acc"] - table["test_acc"]).idxmax())
best_k = int(table["test_acc"].idxmax())
k1_regions = int(table.loc[1, "n_regions"])
two_region_k = int(table.index[(table["n_regions"] == 2).to_numpy().argmax()])
print(f"k with the maximum train-test gap: {overfit_k} | best k on the test set: {best_k} | regions at k=1: {k1_regions} | first k with two regions: {two_region_k}")

In [ ]:
roughness_decreases = bool(table["roughness"].is_monotonic_decreasing)
train_below_test_at_large_k = bool((table["train_acc"] < table["test_acc"]).any())
k1_has_train_one = bool(table.loc[1, "train_acc"] == 1.0)
print("decreasing roughness:", roughness_decreases, "| there is a k with train < test:", train_below_test_at_large_k, "| perfect training with k = 1:", k1_has_train_one)

## Check

In [ ]:
with verify():
    check(list(table.columns) == ["train_acc", "test_acc", "roughness", "n_regions"] and list(table.index) == ks, "table with the four columns for nine values of k", "index k, columns train_acc, test_acc, roughness, n_regions")
    G1 = grid_map(1)
    rough_ref = ((G1[:, 1:] != G1[:, :-1]).sum() + (G1[1:, :] != G1[:-1, :]).sum()) / (G1[:, 1:].size + G1[1:, :].size)
    check(close(table.loc[1, "roughness"], rough_ref), "roughness computed as the fraction of different adjacent pairs", "compare G[:, 1:] with G[:, :-1] and G[1:, :] with G[:-1, :]")
    check(table.loc[1, "n_regions"] == label(G1 == 0)[1] + label(G1 == 1)[1], "number of regions: connected components of 0 and of 1", "label(G == 0)[1] + label(G == 1)[1]")
    check(overfit_k == (table["train_acc"] - table["test_acc"]).idxmax() and overfit_k == 1, "the maximum train-test gap is at k = 1", "idxmax of the difference")
    check(best_k == table["test_acc"].idxmax() and 9 <= best_k <= 61, "the best k on the test set is intermediate (between 9 and 61)", "idxmax of test_acc")
    check(k1_regions >= 8 and two_region_k <= 15 and table.loc[201, "n_regions"] == 2, "with k = 1 at least 8 regions; from k = 15 at most two; with k = 201 exactly two", "n_regions in table")
    check(roughness_decreases is False and table.loc[1, "roughness"] > 1.5 * table.loc[15, "roughness"] and abs(table.loc[15, "roughness"] - table.loc[61, "roughness"]) < 0.002, "the roughness does NOT always fall: it drops until k = 15 (at k = 1 it is over 1.5 times) and then stays on a plateau", "look at the roughness column: is_monotonic_decreasing")
    check(k1_has_train_one is True and train_below_test_at_large_k is False and abs(table.loc[201, "train_acc"] - table.loc[201, "test_acc"]) < 0.02 and table.loc[201, "test_acc"] < table["test_acc"].max() - 0.08, "k = 1: perfect training; the training is never below the test; at k = 201 train and test coincide but are both low (underfitting)", "compare train_acc and test_acc row by row")
done()

## What you learned

- **The number of regions is a complexity gauge**: with `k = 1` the map has 11 regions (the two moons plus nine islands created by the noisy points), with `k = 3` five, with `k = 5` four and from `k = 9` two remain, the two moons. The roughness falls by more than a third
  between `k = 1` and `k = 15` (0.018 against 0.009) and then stays on a **plateau**: when the boundary is the true one, increasing `k` no longer shortens it, it only moves it.
- The **train-test gap** is maximal at `k = 1` (training 1.00, test 0.88): it is the classic symptom of overfitting and is read without plots.
- The best `k` on the test set is intermediate (31 out of 300 examples, but between 9 and 61 the accuracy is practically equal: 0.88-0.89): a boundary neither jagged nor too smooth. With `k = 201` (2/3 of the training set) the model becomes almost a line and collapses to 0.77.
- At `k = 201` training and test coincide (0.77 and 0.77) but are **both low**: it is underfitting, not good generalization. A small gap between training and test is good news only if the two accuracies are high.

## Common pitfall

Choosing `k` "by eye" from the boundary plot: a smooth boundary is pretty, but the only judge is validation. The numeric measures (regions, roughness) help you understand, not choose.

## Interview question

<details><summary>How do you explain to a colleague that `k` regulates the complexity?</summary>

With `k = 1` the prediction at a point depends only on the nearest neighbor: the boundary follows every example and every error, it has dozens of regions and the training is perfect. As `k` grows the prediction is an average over more points, the boundary simplifies and in the end
it becomes the most frequent class. It moves the model along the bias-variance tradeoff; I choose it with cross-validation.

</details>